<a href="https://colab.research.google.com/github/bmulaw/NLP-Notebooks/blob/main/NLP_TP2_Embeddings_LexRank_IMDb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP2 — Embeddings, LexRank and entity opinion

**Duration: 3 hours · Language: English · Environment: Google Colab**

In TP1, you compared sentences using word overlap and analysed opinions with dictionaries. Here, you will use pretrained word vectors, build an extractive summarizer, and apply pretrained neural models to IMDb reviews.

Work through the sections in order. Run each cell before moving on; complete the `TODO` cells yourself. Explain your observations in the notebook.

| Part | Format | Suggested time |
|---|---|---:|
| Word and sentence embeddings | Guided | 25 min |
| Word Mover's Distance | Semi-guided | 20 min |
| LexRank summarization | Autonomous work | 45 min |
| IMDb: named entities and sentiment | Guided | 25 min |
| Entity opinion and summaries | Autonomous work | 65 min |

In Colab, choose **Runtime → Change runtime type → T4 GPU** if available. CPU works on a small sample but is slower. The first run downloads the GloVe vectors, the IMDb dataset and two models; an internet connection and a few minutes are needed. No training is required.

In [ ]:
%pip install --only-binary=:all: "gensim==4.4.0" "nltk==3.9.2" "POT>=0.9.4" "datasets==4.3.0" "transformers==4.46.3" "accelerate" "safetensors"

In [2]:
import html
import re
from collections import Counter

import numpy as np
import nltk
import gensim.downloader as api
import torch
from datasets import load_dataset
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.cluster.util import cosine_distance
from transformers import pipeline

for resource in ["punkt", "punkt_tab", "stopwords"]:
    nltk.download(resource, quiet=True)
stop_words = set(stopwords.words("english"))
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")

Device: CPU


## 1. Static word embeddings — guided

GloVe associates a fixed vector with each word. Words occurring in similar contexts tend to have vectors pointing in similar directions. We use the 50-dimensional `glove-wiki-gigaword-50` model (about 66 MB). Gensim downloads and caches it. A word has the same vector in every sentence, regardless of its meaning in context.

In [3]:
vectors = api.load("glove-wiki-gigaword-50")
print("Vector dimension:", vectors.vector_size)
print("First 10 coordinates of 'cat':", vectors["cat"][:10])
print("Vocabulary size:", len(vectors))
print("Similarity cat / dog:", vectors.similarity("cat", "dog"))
print("Similarity cat / book:", vectors.similarity("cat", "book"))
print("Words close to 'movie':", vectors.most_similar("movie", topn=5))

[==================================================] 100.0% 66.0/66.0MB downloaded
Vector dimension: 50
First 10 coordinates of 'cat': [ 0.45281  -0.50108  -0.53714  -0.015697  0.22191   0.54602  -0.67301
 -0.6891    0.63493  -0.19726 ]
Vocabulary size: 400000
Similarity cat / dog: 0.9218005
Similarity cat / book: 0.43516466
Words close to 'movie': [('movies', 0.9322481155395508), ('film', 0.9310100078582764), ('films', 0.8937394618988037), ('comedy', 0.8902585506439209), ('hollywood', 0.8718216419219971)]


**Discuss:** Which pair is closer? Do the nearest words always have exactly the same meaning? Try two other pairs, including an unrelated pair. What happens if a word is absent from the vocabulary?

In [4]:
# Try your examples here.
print("cat is in the vocabulary:", "cat" in vectors)
print("made-up word is in the vocabulary:", "fluffyxyzabc" in vectors)

print("Similarity paris / achilles:", vectors.similarity("paris", "achilles"))
print("Similarity paris / fashion:", vectors.similarity("paris", "fashion"))
print("Words close to 'paris':", vectors.most_similar("paris", topn=5))

cat is in the vocabulary: True
made-up word is in the vocabulary: False
Similarity paris / achilles: -0.100913756
Similarity paris / fashion: 0.5577139
Words close to 'paris': [('prohertrib', 0.8611263036727905), ('france', 0.8025329113006592), ('brussels', 0.7796469926834106), ('amsterdam', 0.7769756317138672), ('french', 0.7736119031906128)]


## 2. Sentence embeddings — guided

We reuse TP1's small corpus. One simple sentence representation is the **mean** of its known word vectors, after NLTK tokenization, lowercasing and stop-word removal. This loses word order and is not a contextual sentence model. Unknown words are ignored; a sentence with no known words receives a zero vector. The cosine of a zero vector is undefined; here we report 0 by convention.

In [5]:
sentences = [
    "The cat chases the mouse.", "The dog chases the ball.",
    "The cat likes milk.", "The dog likes milk.",
    "The child reads a book.", "The child likes the book.",
    "The bird sings in the tree.", "The cat sleeps in the tree.",
    "The dog sees the dog.", "The child reads a story.",
]

def content_tokens(text):
    """Use NLTK to keep lowercase alphabetic content words."""
    return [word.lower() for word in word_tokenize(text)
            if word.isalpha() and word.lower() not in stop_words]

def sentence_embedding(text):
    """Average known GloVe word vectors; return zeros if no word is known."""
    known = [vectors[word] for word in content_tokens(text) if word in vectors]
    if not known:
        return np.zeros(vectors.vector_size)
    return np.mean(known, axis=0)

def sentence_cosine(a, b):
    """Compute cosine similarity via NLTK; use 0 when a vector is zero."""
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0.0
    # NLTK returns distance = 1 - similarity.
    return float(1 - cosine_distance(a, b))

for i in [0, 1, 4]:
    print(i, sentences[i], content_tokens(sentences[i]))
print("D0 / D1:", sentence_cosine(sentence_embedding(sentences[0]), sentence_embedding(sentences[1])))
print("D0 / D4:", sentence_cosine(sentence_embedding(sentences[0]), sentence_embedding(sentences[4])))

0 The cat chases the mouse. ['cat', 'chases', 'mouse']
1 The dog chases the ball. ['dog', 'chases', 'ball']
4 The child reads a book. ['child', 'reads', 'book']
D0 / D1: 0.8340287208557129
D0 / D4: 0.43401896953582764


**Discuss:** Compare these results with the Jaccard similarities from TP1. Why might sentences with different words still be similar? What information does averaging discard?

- *TODO*

## 3. Word Mover's Distance — semi-guided

Word Mover's Distance (WMD) measures the minimum cost of transporting the word distribution of one sentence to the other, using distances between pretrained word vectors as transport costs. A **smaller distance** means sentences are closer. It does not use word order. This is a distance and not a similarity.

Use `content_tokens` above. Look in the **Gensim KeyedVectors** documentation for a method that computes WMD from two lists of tokens; look in **Python's `math`** module for a way to check whether a number is finite. The `POT` package installed above supplies the optimal-transport solver. Gensim ignores words outside its vocabulary; if either sentence has no known word the distance is infinite. Handle that case explicitly.

Complete `word_mover_distance(text_a, text_b, model)`:

- `text_a` and `text_b` are the two original sentences to compare, each given as a Python string (for example, `sentences[0]` and `sentences[1]`).
- `model` is a Gensim `KeyedVectors` object containing the pretrained word embeddings. In this notebook, pass `vectors`, the GloVe model loaded in section 1. Use this parameter to check the vocabulary and compute the distance.

Compare D0 with D1 and D4, then compare two sentences of your choice. Explain whether the ranking agrees with cosine. Documentation: [Gensim KeyedVectors](https://radimrehurek.com/gensim/models/keyedvectors.html), [Python math](https://docs.python.org/3/library/math.html).

In [6]:
import math

def word_mover_distance(text_a, text_b, model):
    """Return the WMD between two sentences using pretrained word embeddings.

    text_a and text_b: original sentences as strings.
    model: Gensim KeyedVectors containing pretrained word embeddings (e.g. vectors).
    Return infinity if either sentence has no word in the model vocabulary.
    """
    tokens_a = content_tokens(text_a)
    tokens_b = content_tokens(text_b)
    # TODO: keep only words present in the vocabulary of model.
    # If either sentence has no known words, return infinity.
    # TODO: find and call the Gensim method on model with the two token lists.
    wmd = model.wmdistance(tokens_a, tokens_b)
    return math.inf if math.isinf(wmd) else wmd

# Example call after completing the function:
# TODO: print and compare the distances; use math to check for infinity.
distance = word_mover_distance(sentences[0], sentences[1], vectors)
distance2 = word_mover_distance(sentences[4], sentences[6], vectors)
print(distance)
print(distance2)

0.4822667892211423
1.0837322636027176


## 4. Extractive summarization with LexRank (Autonomous work)

We will rank sentences in a graph and extract the most central ones. **Extractive** means that the output contains original sentences, without rewriting them. Use the ten TP1 sentences first. Afterwards you will reuse this system for IMDb.

### Algorithm

1. Compute an embedding for each sentence and its pairwise cosine similarities. Build an `n × n` similarity matrix `S`: clip negative values to zero, remove self-links (`S[i, i] = 0`), and set values below threshold `0.5` to zero. Keep remaining similarity values as **weighted** edges. This is one weighted LexRank variant.
2. Turn `S` into a row-stochastic matrix `P`: divide each nonzero row by its sum. If a row contains only zeros, replace it with `1/n` in every column. `P[i, j]` is the probability of moving from sentence i to sentence j.
3. Start with a uniform score vector `r = [1/n, …, 1/n]`. Repeatedly update `r_new = (1-d)/n + d * P.T @ r`, with `d = 0.85`, until the sum of absolute changes is below `1e-8` or 200 iterations have elapsed. Report convergence. Scores should sum to 1.
4. Choose the `N` largest scores; break ties using the original index. Display the selected original sentences in their original order for readability, alongside their scores.

For zero sentences return an empty result; for one sentence return score 1. Use NumPy. Do not use a ready-made LexRank implementation: implement the matrix and updates. Thresholding may produce an edgeless graph; the dangling-row rule handles it. Consider whether repeated or very similar sentences can make the summary redundant.

In [46]:
def similarity_matrix(texts, threshold=0.5):
    """Return a symmetric weighted graph with no self-links."""
    # TODO: compute embeddings, all pairwise cosines, clip and threshold.
    matrix_pair = []
    for i in range(len(texts)):
      scores = []
      for j in range(len(texts)):
        if i != j:
          sentence_i = sentence_embedding(texts[i])
          sentence_j = sentence_embedding(texts[j])
          cos_distance = sentence_cosine(sentence_i, sentence_j)
          cosine_pairing = cos_distance if cos_distance > 0.5 else 0
        else:
          cosine_pairing = 0
        scores.append(cosine_pairing)
      matrix_pair.append(scores)
    print(np.array(matrix_pair))
    return np.array(matrix_pair)

def lexrank_scores(similarity, damping=0.85, tolerance=1e-8, max_iter=200):
    """Return scores and a convergence flag for a square similarity matrix."""
    # TODO: normalize rows; treat zero rows as uniform transitions.
    # TODO: iterate the rank update and return (scores, converged).
    raise NotImplementedError("Implement extractive summarization.")

def summarize(texts, n=3, threshold=0.5):
    """Return (original index, sentence, score) tuples in original order."""
    # TODO: handle empty input, compute scores, choose the top n with tie-breaking.
    raise NotImplementedError("Implement extractive summarization.")

In [47]:
# Run these checks after implementing your functions.
S = similarity_matrix(sentences)
assert S.shape == (10, 10) and np.allclose(S, S.T)
assert np.allclose(np.diag(S), 0) and np.all(S >= 0)
scores, converged = lexrank_scores(S)
assert len(scores) == 10 and np.isclose(sum(scores), 1) and np.all(scores >= 0)
assert np.allclose(lexrank_scores(np.zeros((3, 3)))[0], np.ones(3) / 3)
assert np.allclose(lexrank_scores(np.zeros((1, 1)))[0], [1])
assert summarize([], n=3) == []
summary = summarize(sentences, n=3)
assert len(summary) == 3 and len({item[0] for item in summary}) == 3
assert [item[0] for item in summary] == sorted(item[0] for item in summary)
print("Converged:", converged)
for index, sentence, score in summary:
    print(f"D{index}: {score:.4f} | {sentence}")

[[0.         0.83402872 0.68051279 0.64421958 0.         0.50567728
  0.67095107 0.7782892  0.80405962 0.        ]
 [0.83402872 0.         0.69170707 0.70687288 0.         0.57038105
  0.58986723 0.6798256  0.80829448 0.50409853]
 [0.68051279 0.69170707 0.         0.98714393 0.56641209 0.71545744
  0.70149255 0.74823177 0.83959126 0.53876239]
 [0.64421958 0.70687288 0.98714393 0.         0.58226335 0.72505498
  0.67074674 0.72025955 0.85805184 0.54994643]
 [0.         0.         0.56641209 0.58226335 0.         0.91358405
  0.60787904 0.50238001 0.66742861 0.97776371]
 [0.50567728 0.57038105 0.71545744 0.72505498 0.91358405 0.
  0.65207613 0.5498867  0.74449426 0.89625394]
 [0.67095107 0.58986723 0.70149255 0.67074674 0.60787904 0.65207613
  0.         0.8274489  0.72272122 0.61170399]
 [0.7782892  0.6798256  0.74823177 0.72025955 0.50238001 0.5498867
  0.8274489  0.         0.78832972 0.53583223]
 [0.80405962 0.80829448 0.83959126 0.85805184 0.66742861 0.74449426
  0.72272122 0.788329

TypeError: cannot unpack non-iterable NoneType object

**Questions:** Which sentences are central? Change the threshold to `0.3` and `0.7`. What happens to the graph and summary? Explain one limitation of applying a summary to this collection of unrelated short sentences.

## 5. IMDb and Hugging Face models — guided

IMDb contains 25,000 labelled training reviews and 25,000 test reviews. As in TP1, we use the training split, preserve case and remove HTML tags. Each review is a document. We use pretrained models for inference; the dataset labels are not model predictions.

We use `dslim/bert-base-NER` for named entities and DistilBERT fine-tuned on SST-2 for positive/negative sentiment. We apply each model to the **review text**. Each tokenizer has a maximum input length, so we explicitly keep the beginning of a review when needed. The models can have different tokenizers and therefore receive different prefixes. We record every affected sentence in `truncation_log.txt`, separately for each model. Neither model sees the discarded part of the review.

### What is a Hugging Face pipeline?

Hugging Face provides the **Hub**, where models and datasets are shared, and the **Transformers** Python library, which loads and runs models. A `pipeline` is a ready-to-use interface for a particular task. We create it once, then call it on text such as `ner(text)` or `sentiment(text)`.

For these NLP tasks, the pipeline connects three stages:

1. **Preprocessing:** the model's tokenizer transforms the input string into tokens and numerical IDs, adds the special tokens expected by the model, and prepares tensors. Tokens can be word pieces rather than whole words. Each model uses its own compatible tokenizer.
2. **Inference:** the pretrained neural model processes those tensors and produces numerical predictions. In this TP, inference uses existing model weights; it does not train or fine-tune the model.
3. **Postprocessing:** the pipeline converts model outputs into readable results. For NER, it groups predicted tokens into entities and returns their positions; for sentiment, it returns a class label and a confidence score.

The first time you create these pipelines, the model files and their tokenizers are downloaded from the Hub and cached. Loading can take a few minutes. We clean IMDb HTML ourselves before passing text to the pipelines. We also truncate reviews explicitly using `truncate_review`, so that affected sentences are recorded in a log; the pipeline does not write this log for us.

### Reading the pipeline parameters

The next cell constructs two pipelines. The arguments mean:

- `"token-classification"`: the task for the NER model. It predicts a class for each token, such as whether it belongs to a person or location.
- `"sentiment-analysis"`: the task for the sentiment model. It classifies the input text as positive or negative with the chosen model.
- `model=...`: the identifier of a pretrained model on the Hugging Face Hub. It selects its weights and configuration; in these calls, the corresponding tokenizer is also loaded automatically. Here we use `dslim/bert-base-NER` and `distilbert/distilbert-base-uncased-finetuned-sst-2-english`.
- `aggregation_strategy="simple"`: an option for the token-classification pipeline. It combines consecutive token predictions into entity spans when their labels are compatible, so a name can be returned as one entity rather than separate pieces. It does not merge different mentions across a review.
- `device=device`: tells the pipeline where to run the model. The preceding line selects `0` for the first CUDA GPU if available, or `-1` for the CPU. This changes the execution device, not the task.

`ner` and `sentiment` are the resulting pipeline objects. Calling either object with a string executes the three stages above. The `clean_review` and `truncate_review` functions in the cell are helpers written for this TP; they are not parameters of `pipeline`.

Read more: [Hugging Face pipeline tutorial](https://huggingface.co/learn/llm-course/chapter1/3) and [pipeline reference](https://huggingface.co/docs/transformers/main_classes/pipelines).

In [ ]:
imdb = load_dataset("stanfordnlp/imdb", split="train")
print("Reviews:", len(imdb), "Fields:", imdb.column_names)

def clean_review(text):
    """Remove HTML tags and normalize whitespace while preserving capitalization."""
    text = html.unescape(text)
    return re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", text)).strip()

device = 0 if torch.cuda.is_available() else -1
ner = pipeline("token-classification", model="dslim/bert-base-NER",
               aggregation_strategy="simple", device=device)
sentiment = pipeline("sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english", device=device)

# Start a fresh log for this notebook run; subsequent calls append to it.
LOG_FILE = "truncation_log.txt"
with open(LOG_FILE, "w", encoding="utf-8") as log:
    log.write("")

def truncate_review(text, doc_id, classifier, model_name, log_path=LOG_FILE):
    """Return the prefix that fits a model and log affected sentence token counts.

    doc_id identifies the IMDb row. Sentence numbers start at 0. Counts exclude
    special tokens and use the current model's tokenizer on the complete review.
    """
    tokenizer = classifier.tokenizer
    # Offsets connect each model token to its character positions in the review.
    encoded = tokenizer(text, add_special_tokens=False, truncation=False,
                        return_offsets_mapping=True)
    offsets = encoded["offset_mapping"]
    budget = tokenizer.model_max_length - tokenizer.num_special_tokens_to_add(pair=False)
    if len(offsets) <= budget:
        return text
    if budget <= 0:
        raise ValueError("The model has no room for text tokens.")
    cutoff = offsets[budget - 1][1]
    # NLTK supplies character spans for the original sentences.
    spans = nltk.data.load("tokenizers/punkt/english.pickle").span_tokenize(text)
    with open(log_path, "a", encoding="utf-8") as log:
        for sentence_id, (start, end) in enumerate(spans):
            original = sum(1 for left, right in offsets if right > start and left < end)
            retained = sum(1 for left, right in offsets[:budget] if right > start and left < end)
            if retained < original:
                log.write(f"sentence {sentence_id} from doc {doc_id} truncated from "
                          f"{original} to {retained} [model: {model_name}]\n")
    return text[:cutoff]

demo = "Tom Hanks gives a wonderful performance in this film."
print("Entities:", ner(demo))
print("Sentiment:", sentiment(demo))

### Inspect the preprocessing

The pipeline exposes its tokenizer as `ner.tokenizer`. The next cell shows the word pieces and input IDs produced for the same short text. `convert_ids_to_tokens` also displays special tokens added for the model. These IDs are vocabulary indices, not the 50-dimensional GloVe vectors used earlier; the transformer uses its own learned representations internally. Compare the displayed tokens with the original words.

In [ ]:
example_text = "Tom Hanks gives a wonderful performance."
print("Text:", example_text)
print("Word pieces:", ner.tokenizer.tokenize(example_text))
encoded_example = ner.tokenizer(example_text)
print("Input IDs:", encoded_example["input_ids"])
print("Tokens including special tokens:", ner.tokenizer.convert_ids_to_tokens(encoded_example["input_ids"]))
print("Attention mask:", encoded_example["attention_mask"])

The **attention mask** identifies which input positions are actual tokens (`1`) rather than padding (`0`). This example contains one unpadded text, so its mask consists of ones. When texts are processed in batches, padding can make their lengths equal. The NER and sentiment tokenizers need not split the same text into identical pieces.

### Read the predictions

In the printed NER output, `entity_group` is the predicted entity type (for example `PER`), `word` is the tokenizer's reconstructed entity, `score` is the model's confidence for that predicted group, and `start` / `end` are character offsets in the input string. For sentiment, `label` is the predicted class and `score` is confidence in that **predicted** class. Neither confidence is guaranteed to be a calibrated probability of correctness. Examine the output of the previous cell and the demo above before continuing.

The NER pipeline returns character offsets `start` and `end` into the **text passed to the model**. Use `ner_text[start:end]` to recover the entity as written. `aggregation_strategy="simple"` combines adjacent subword predictions. The sentiment pipeline returns a predicted label and its confidence, not automatically the probability of the positive class. We define a signed score: positive confidence for `POSITIVE`, negative confidence for `NEGATIVE`.

The log records lines such as `sentence 4 from doc 12 truncated from 18 to 7 [model: NER]`. Sentence and document IDs start at 0; token counts exclude special tokens and depend on the model's tokenizer. A count of 0 means the whole sentence falls beyond that model's input limit. An unaffected document produces no log line. Re-running the setup cell clears the log.

The models may miss names or misclassify sentiment. A review-level sentiment score describes the model's prediction for the retained prefix of the whole review; it does not necessarily describe the opinion expressed about every entity mentioned in it.

In [ ]:
doc_id = 0
review = clean_review(imdb[doc_id]["text"])
ner_text = truncate_review(review, doc_id, ner, "NER")
sentiment_text = truncate_review(review, doc_id, sentiment, "sentiment")
entities = ner(ner_text)
result = sentiment(sentiment_text)[0]
names = [(ner_text[e["start"]:e["end"]], e["entity_group"]) for e in entities]
signed = result["score"] if result["label"] == "POSITIVE" else -result["score"]
print("Original review:", review[:300], "...")
print("Entities in the NER prefix:", names)
print("Sentiment of the sentiment-model prefix:", result, "signed:", round(signed, 3))
print("Truncation log:", LOG_FILE)

## 6. Entity opinion and representative sentences (Autonomous work)

Build the TP1 chain using the new tools. **The unit for sentiment is a review.** Search the complete IMDb training split for the exact name, ignoring case and allowing multiple spaces; then process at most the first 30 matching reviews to bound runtime. The results describe this selected sample. Do not use the dataset's reference labels to score an entity.

1. Select an entity such as `Tom Hanks`. Find matching reviews with a whole-phrase regular expression; `Tom Hankson` must not match. Keep original dataset row IDs.
2. For each selected review, clean its text and call `truncate_review` once for each model, with the review's dataset row ID and a distinct model name. Run NER on its truncated **review text**. Retain a review only if a predicted entity's text, obtained using NER offsets, equals the complete query ignoring case and repeated whitespace. Count each review once. Names occurring only in the discarded text cannot be detected; aliases are not resolved.
3. Run sentiment on the review's sentiment-model prefix. Save the row ID, predicted label, confidence and signed score. Count positive and negative reviews. Calculate the mean signed score in [−1, 1] across retained reviews, once per review. This is sentiment for the review prefix, not necessarily sentiment about the entity.
4. Split each retained review into sentences with NLTK. Collect original sentences containing an NER-confirmed occurrence, using the NER offsets to identify the corresponding sentence. Pass these sentences to your LexRank implementation and extract up to 3 representative sentences with their rank scores. A sentence can extend beyond the NER prefix; the entity itself must have been detected in the prefix. Repeated sentences can dominate; report this limitation.
5. Print the number of full-split matches, processed reviews and retained reviews, and show where the truncation log was saved. If there are no retained reviews or sentences, display a message and skip the corresponding mean or summary; do not invent examples.

Test your functions on small examples. For a different entity or more data, change the constants below. Models can receive different review prefixes: a detected entity can lie outside the portion used for sentiment.

In [ ]:
ENTITY = "Tom Hanks"
MAX_REVIEWS = 30

def normalize_name(name):
    """Ignore case and repeated whitespace when comparing full entity names."""
    return " ".join(name.casefold().split())

def name_pattern(name):
    """Match the whole name, allowing different amounts of whitespace."""
    words = name.split()
    if not words:
        raise ValueError("The entity name must not be empty.")
    return re.compile(r"(?<!\w)" + r"\s+".join(map(re.escape, words)) + r"(?!\w)", re.I)

assert name_pattern("Tom Hanks").search("tom   hanks acts.")
assert not name_pattern("Tom Hanks").search("Tom Hankson acts.")

def candidate_review_ids(dataset, entity, limit):
    """Return (all matching row IDs, selected row IDs), in dataset order."""
    # TODO: search cleaned review text using name_pattern; select at most limit.
    raise NotImplementedError("Find candidate reviews.")

def entity_review_records(dataset, row_ids, entity):
    """Analyse reviews and return one record per NER-confirmed review.

    Each record: {'row_id': int, 'label': str, 'confidence': float,
                  'signed_score': float, 'sentences': list of str}.
    """
    # TODO: truncate each cleaned review separately for NER and sentiment.
    # TODO: use NER offsets and normalize_name to check the complete entity.
    # TODO: score each retained review once with the sentiment model.
    # TODO: use NLTK sentence spans and NER offsets to collect matching sentences.
    raise NotImplementedError("Analyse matching reviews.")

def entity_report(records, n=3):
    """Print review counts, mean signed sentiment and a LexRank sentence summary."""
    # TODO: handle empty records; print counts, mean and representative sentences.
    raise NotImplementedError("Report the entity opinion and summary.")

In [ ]:
# Test whole-phrase matching without calling the neural models.
toy = [{"text": "Tom Hanks is here."}, {"text": "Tom Hankson is here."},
       {"text": "TOM   HANKS is back."}]
all_ids, selected_ids = candidate_review_ids(toy, "Tom Hanks", 1)
assert all_ids == [0, 2] and selected_ids == [0]

all_ids, selected_ids = candidate_review_ids(imdb, ENTITY, MAX_REVIEWS)
print("Matching reviews in the full split:", len(all_ids))
print("Selected reviews:", len(selected_ids))
records = entity_review_records(imdb, selected_ids, ENTITY)
print("NER-confirmed reviews:", len(records))
print("Truncation log:", LOG_FILE)
entity_report(records, n=3)

### Interpretation

1. Does the mean score describe the entity itself, or the sentiment of review prefixes mentioning it? Explain with an example.
2. Inspect at least two predictions. Find an NER or sentiment error if possible.
3. Compare the TP1 lexicon-based approach with the transformer models. Discuss one advantage and one limitation of each.
4. How would more reviews, aliases, repeated sentences or a different entity affect the result?

**Submission:** save the notebook with your code, printed results and written answers. List any exercise you could not complete.